In [6]:
# Make sure to import packages and run code from the initial_setup notebook.

# The below code will run a GaussianProcessRegressor surrogate model using an Expected Improvement acquisition function.
# You can swap out the RBF kernel for a Matern kernel, but I found RBF fits better for most of the functions in this project.
# If you want to test which kernel fits better, choose the result with the highest log_marginal_likelihood_value.

# Generating default random value for consistency.
rng = np.random.default_rng(19)

# In this example, we'll use Function 1. To run this analysis on a different function, simply swap it out.
# For example, insert df_function_5 instead of df_function_1.
data = df_function_1

# Creating X as a dataframe of input values. To use a function with more dimensions than 2, simply add these X-values.
# For example, for df_function_5 with 4 dimensions, X becomes X = data[['X_1', 'X_2', 'X_3', 'X_4']].values.
X = data[['X_1', 'X_2']].values

# y stays the same for all functions.
y = data['y'].values

# Aggressive output scaling for Function 1. For Functions 2-8, I used the yeo-johnson power transformation below as recommended by HEBO. 
# When running Functions 2-8, it's recommended to comment out this aggressive scaling as it will negatively impact your results.
alpha = 0.02
y_eng = np.sign(y) * np.power(np.abs(y), alpha)

# Scale y values for Functions 2-8. 
# pt = PowerTransformer(method='yeo-johnson')
# y_eng = pt.fit_transform(y.reshape(-1, 1))

# Set this for number of input variables. For example, if running Function 5, num_dimensions = 4.
num_dimensions = 2

# Fitting an RBF kernel plus a WhiteKernel for noise assumption.
# Setting length_scale=np.ones(num_dimensions) allows the length_scale for each dimension to be different.
kernel = (RBF(length_scale=np.ones(num_dimensions), length_scale_bounds=(1e-4, 1e4))
          + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-30, 1.0)))

# Using a GaussianProcessRegressor.
model = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-20, # Assume some non-zero noise level for numerical stability
    n_restarts_optimizer=500,
    normalize_y=True,
    random_state=19
)
model.fit(X, y_eng)

# Displaying the kernel length scales and log marginal likelihood (higher is better).
print(f'Fitted kernel: {model.kernel_}')
print(f'Log marginal likelihood: {model.log_marginal_likelihood_value_}')

# Segregate the fitted components
rbf_fitted = model.kernel_.k1
noise_fitted = model.kernel_.k2.noise_level
print(f'Learned noise level (normalized units): {noise_fitted:.6f}')

# Build a new model with RBF length scales and learned noise assumption.
model = GaussianProcessRegressor(
    kernel=rbf_fitted,
    alpha=noise_fitted,
    optimizer=None, # Don't re-optimize
    normalize_y=True,
    random_state=19
)
model.fit(X, y_eng)

# Display training mean and training standard deviation.
print("Training mean:", model._y_train_mean)
print("Training std:", model._y_train_std)

# Print normalized y values to check what the model is seeing.
y_normalized = (y_eng - model._y_train_mean) / model._y_train_std
print(y_normalized)

# Set f_best equal to current maximum value.
f_best = np.max(y_eng)

# Using the inverse of SciPy's minimize function as the acquisition function.
# Specify exploration/exploitation level here. xi = 0.0 is maximum exploitation.
# Higher xi equals more exploration.
xi = 0.0

def negative_ei(x):
    # scipy.optimize minimizes. To maximize EI, minimize negative EI.
    # Reshaping to a 2D array.
    x = x.reshape(1, -1)

    mu, sigma = model.predict(x, return_std=True)

    mu = mu[0]
    sigma = sigma[0]

    sigma = max(sigma, 1e-12)

    # Finding improvement over current f_best.
    improvement = mu - f_best - xi
    Z = improvement / sigma

    ei = (
        improvement * norm.cdf(Z)
        + sigma * norm.pdf(Z)
    )

    return -ei # Return as a negative to maximize.

# Setting bounds for potential X values as stated in the Capstone rules.
bounds = [(0.0, 0.999999) for _ in range(num_dimensions)]

# Optimize the EI acquisition function. You can make this run quicker by setting n_starts equal to a lower
# number, but 2000 restarts doesn't take very long.
n_starts = 2000
best_ei = -np.inf
best_x = None

for i in range(n_starts):
    # Pick a random starting point within the bounds.
    start_point = rng.uniform(0.0, 0.999999, size=num_dimensions)
    
    # Run the L-BFGS-B optimizer
    res = minimize(
        negative_ei,
        start_point,
        bounds=bounds,
        method='L-BFGS-B'
    )
    
    # Multiply by -1 to get a positive result back. This checks if the latest run found a higher EI than previous runs.
    current_ei = -res.fun

    if current_ei > best_ei:
        best_ei = current_ei
        best_x = res.x

# Format the array.
formatted_X = [f"{val:.6f}" for val in best_x]

# Predict the final mean and standard deviation for the chosen point.
final_mean_eng, final_std_eng = model.predict(best_x.reshape(1, -1), return_std=True)

# Inverse transform to convert engineered output back to original if using aggressive scaling for Function 1.
# If using yeo-johnson for Functions 2-8, comment this line out.
pred_y_orig_scale = np.sign(final_mean_eng[0]) * np.power(np.abs(final_mean_eng[0]), 1 / alpha)

# If using the yeo-johnson transformation for Functions 2-8, inverse_transform.
# pred_y_orig_scale = pt.inverse_transform(final_mean_eng.reshape(-1, 1))[0, 0]

print(f"Next points to sample: {formatted_X}")
print(f"Predicted Mean (Original Scale):   {pred_y_orig_scale:.6e}")

Fitted kernel: RBF(length_scale=[0.147, 0.0278]) + WhiteKernel(noise_level=5.13e-18)
Log marginal likelihood: -22.606055587372047
Learned noise level (normalized units): 0.000000
Training mean: 0.14569035620092266
Training std: 0.44180525258647885
[-0.26989301 -0.05745888  0.79876355 -0.32208348 -2.3523605  -0.52094709
 -0.36452857  0.03570595 -0.27405426 -0.07233176 -0.32783124  1.26989292
 -0.46838498 -0.2770196  -0.44889482 -0.33151752 -0.0113622   1.27758012
  1.4469917  -2.37710417  1.19540248  1.11509272  1.33634262]
Next points to sample: ['0.440910', '0.358236']
Predicted Mean (Original Scale):   3.179093e-03
